# Getting cutouts straight from the Zooniverse `.csv` export
The JSON files from Sophie's paper contain aggregated data from ~20000 volunteers.

We want to get at the "raw" box data. So we do that here.

A UMN researcher (Lestat) exported the Zooniverse "box the jet" information from the website to [a CSV file](https://drive.google.com/file/d/16A91wIPhhaEGvMGp0hrOeVNCsu-vwvVY/view?usp=sharing).
It contains information on the images which were boxed so that the bounding boxes can be converted from image to physical coordinates.
It was exported using `bash`:
```bash
# First, install the Zooniverse panoptes package
$(which uv) pip install panoptescli

# Configure it with username/password which has access to the SJH project
panoptes configure

# Finally, download the "box the jet" workflow to a CSV
panoptes workflow download-classifications 21225 box-the-jets.csv
# Optionally regenerate the workflow data set if required by adding the -g option
# panoptes workflow download-classifications -g 21225 box-the-jets.csv
```

There is also additional information regarding the "base" of each jet.
We don't care about the bases of the jets, just the bounding boxes, so we can ignore that.

In [1]:
import astropy.units as u
import astropy.coordinates as acoord
import regions

import zooniverse_processing as zp

In [2]:
cutoff_version = 50.63
extracted = zp.load_zooniverse_csv("box-the-jets.csv", cutoff_version=cutoff_version)

In [3]:
for id_, e in extracted.items():
    stime: str = e.meta["time"]["start_time"]
    if "2012" in stime:  # stime.startswith('2012-09'):
        print(e.meta["time"])
    # print(stime)

In [4]:
# Test out converting the Zooniverse box centers into helioprojective arcseconds

test_id = 97654458
zooniverse_pair = extracted[test_id]
meta_ = extracted[test_id].meta
bounding_boxes = extracted[test_id].bounding_boxes
meta_

{'fits_header': {'naxis1': 200.0,
  'naxis2': 200.0,
  'cunit1': 'arcsec',
  'cunit2': 'arcsec',
  'crval1': 0.0,
  'crval2': 0.0,
  'cdelt1': 0.600165,
  'cdelt2': 0.600165,
  'crpix1': -1385.199951,
  'crpix2': 579.160034,
  'crota2': -0.131591},
 'image_extract_data': {'lower_left_x_prop': 0.22375,
  'lower_left_y_prop': 0.1099999999999999,
  'upper_right_x_prop': 0.80125,
  'upper_right_y_prop': 0.88,
  'width': 960.0,
  'height': 720.0000000000001},
 'time': {'start_time': '2015-08-29T22:06:42.120000Z',
  'end_time': '2015-08-29T22:26:18.120000Z'},
 'frame_filenames': []}

In [5]:
box_files = zp.reassociate_bounding_boxes(meta_, bounding_boxes)

In [6]:
len(box_files), len(bounding_boxes)

(12, 12)

In [9]:
box_files[0].absolute()

PosixPath('/home/william/grad_school/glesener/analysis/jethunter/helioml-jets/data/2015/aia.lev1_euv_12s.2015-08-29T222032Z.304.image_lev1.fits')

In [ ]:
# Test getting a sky region from the Zooniverse data
jet_regions = tuple(
    zp.sky_region_from_zooniverse_rect(box=b, meta=meta_) for b in bounding_boxes
)

In [ ]:
lower_left, upper_right = zooniverse_pair.bounding_corners_from_boxes()

In [ ]:
import sunpy.map as smap
from sunpy.map.sources import sdo
from sunpy.coordinates import SphericalScreen
import matplotlib.pyplot as plt

%matplotlib qt

plt.style.use("../nice.mplstyle")

m: sdo.AIAMap = smap.Map("2015/aia.lev1_euv_12s.2015-08-29T222608Z.304.image_lev1.fits")
fig, ax = plt.subplots(subplot_kw={"projection": m.wcs}, layout="none")
m.plot(axes=ax)

time_delta = 5 << u.min
with SphericalScreen(center=m.observer_coordinate):
    map_time = m.observer_coordinate.obstime
    for jet_region in jet_regions:
        box_time = jet_region.region.center.obstime
        if False:  # np.abs(map_time - box_time) > time_delta:
            print("skipping region")
            continue
        px = jet_region.region.to_pixel(wcs=m.wcs)
        px.plot(ax=ax, color="pink", alpha=0.7)

    rect = regions.RectangleSkyRegion(
        center=acoord.SkyCoord(
            *(lower_left + upper_right) / 2, frame=m.coordinate_frame
        ),
        width=2 * (upper_right - lower_left)[0],
        height=2 * (upper_right - lower_left)[1],
    )

    goofy = rect.to_pixel(wcs=m.wcs)
    goofy.plot(ax=ax)
    ax.scatter(
        rect.center.Tx.to_value(u.deg),
        rect.center.Ty.to_value(u.deg),
        color="blue",
        zorder=100,
        transform=ax.get_transform("world"),
    )

plt.show()

In [ ]:
from sunpy.net import attrs as a, Fido

time_range = a.Time("2025-09-19T21:45:00Z", "2025-09-19T21:46:00Z")
email = "wilbert.steinbun@gmail.com"
query = Fido.search(
    time_range,
    a.Wavelength(171 << u.angstrom),
    a.jsoc.Series.aia_lev1_euv_12s,
    a.jsoc.Notify(email),
    a.jsoc.Segment.image,
)

In [ ]:
Fido.fetch(query[0][0], path="./")

In [ ]:
import sunpy.map
import numpy as np

smap = sunpy.map.Map("aia.lev1_euv_12s.2025-09-19T214522Z.171.image_lev1.fits")
fig, ax = plt.subplots(frameon=False, layout="constrained")
# Disable the axis
ax.set_axis_off()

# Plot the map.
# Since we are not interested in the exact map coordinates,
# we can simply use :meth:`~matplotlib.Axes.imshow`.
norm = smap.plot_settings["norm"]
norm.vmin, norm.vmax = np.percentile(smap.data, [1, 99.9])
ax.imshow(smap.data, norm=norm, cmap=smap.plot_settings["cmap"], origin="lower")

plt.savefig("aia.png", dpi=300)